# OUMUAMUA - En búsqueda de su estrella madre

## Librerías

In [1]:
%pip install pymcel celluloid astroquery astropy scipy numpy pandas matplotlib -Uq

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.4 which is incompatible.
uxarray 2025.6.0 requires numpy<2.3, but you have numpy 2.4.4 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!



## Estado inicial de referencia (base común)
Extraer un estado heliocéntrico de 1I/'Oumuamua cercano a perihelio y utilizarlo como condición de referencia para todos los experimentos. Según wikipedia, esto representa la época del 14 de octubre de 2017.

In [41]:
epoch_ref = '2017-10-14'  # Fecha de referencia cercana al perihelio

def extrae_estado(resultado):
    if isinstance(resultado, tuple) and len(resultado) >= 3:
        estado = np.asarray(resultado[2], dtype=float)
        if estado.ndim == 2:
            estado = estado[0]
        return estado
    return np.asarray(resultado, dtype=float)

raw = pc.consulta_horizons(
    id='1I',
    location='@0',
    epochs=epoch_ref,
    datos='vectors',
    propiedades=[('x', 'km'), ('y', 'km'), ('z', 'km'), ('vx', 'km/s'), ('vy', 'km/s'), ('vz', 'km/s')]
)

estado_ref = extrae_estado(raw)
r0, v0 = estado_ref[:3], estado_ref[3:]

print('Estado extraído correctamente:')
print('r0 [km]:', r0)
print('v0 [km/s]:', v0)


Estado extraído correctamente:
r0 [km]: [ 1.45267492e+08  7.47620204e+07 -1.07128187e+07]
v0 [km/s]: [44.83537051 10.39892688 14.33813591]


## Caracterización de la hipérbola
Se obtienen elementos orbitales con `estado_a_elementos(mu, estado)` y se valida energía específica positiva, mostrando que el objeto no pertenece a nuestro sistema solar.

In [46]:
mu_sun = pc.constantes.mu_sun  # valor base en SI (m^3/s^2)
mu_km = mu_sun / 1e9           # conversión a km^3/s^2 (consistente con r0, v0)

print('Parámetro gravitacional del Sol [m^3/s^2]:', mu_sun)
print('Parámetro gravitacional del Sol [km^3/s^2]:', mu_km)

# elementos = (p, e, i, Omega, omega, f) según convención de pymcel.
p, e, inc, Omega, omega, f = pc.estado_a_elementos(mu_km, estado_ref)

print('Elementos orbitales extraídos correctamente:')
print('p [km]:', p)
print('e:', e)
print('i [°]:', np.degrees(inc))
print('Omega [°]:', np.degrees(Omega))
print('omega [°]:', np.degrees(omega))
print('f [°]:', np.degrees(f))

Parámetro gravitacional del Sol [m^3/s^2]: 1.3271244004127942e+20
Parámetro gravitacional del Sol [km^3/s^2]: 132712440041.27942
Elementos orbitales extraídos correctamente:
p [km]: 85604594.26472798
e: 1.20554107345583
i [°]: 123.1137532641553
Omega [°]: 24.781488427458203
omega [°]: 242.20374636009197
f [°]: 113.31585543976549


In [47]:
# Validación física rápida para clasificar la cónica
ere = 0.5 * np.dot(v0, v0) - mu_km / np.linalg.norm(r0)
mare = np.linalg.norm(np.cross(r0, v0))


print('Energía específica [km^2/s^2]:', ere)
print('¿Órbita hiperbólica (ε>0: )?', ere > 0)
print('Momento angular específico [km^2/s]:', mare)

Energía específica [km^2/s^2]: 351.3972315370671
¿Órbita hiperbólica (ε>0: )? True
Momento angular específico [km^2/s]: 3370577781.8670444


## Geometría Analítica

"mirar hacia atrás" en el tiempo.
Método Matemático: Utilizando la excentricidad (e) obtenida, calcularás el ángulo de las asíntotas de la hipérbola mediante la relación cosψ=1/e
.
Rutina auxiliar: Implementarás las matrices de rotación de Euler (M(ω,i,Ω)) para orientar el vector de la asíntota de entrada en el sistema de referencia inercial. Esto dirá exactamente desde qué dirección del espacio profundo llegó el objeto.

In [50]:
# Geometría hiperbólica + orientación inercial con Euler M(ω, i, Ω)

# Ángulo de asíntota (según clase): cos(psi)=1/e
psi = np.arccos(1.0 / e)

# Anomalía verdadera al infinito: cos(f_inf) = -1/e
f_inf = np.arccos(-1.0 / e)

def R1(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0],
                     [0, c, -s],
                     [0, s,  c]])

def R3(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[ c, -s, 0],
                     [ s,  c, 0],
                     [ 0,  0, 1]])

# Matriz de rotación perifocal -> inercial
M_euler = R3(Omega) @ R1(inc) @ R3(omega)

# Direcciones asintóticas en marco perifocal (posición)
rhat_in_pf  = np.array([np.cos(-f_inf), np.sin(-f_inf), 0.0])  # entrada (t -> -inf)
rhat_out_pf = np.array([np.cos(+f_inf), np.sin(+f_inf), 0.0])  # salida  (t -> +inf)

# Dirección de velocidad asintótica en perifocal
vhat_in_pf = np.array([-np.sin(-f_inf), e + np.cos(-f_inf), 0.0])
vhat_in_pf = vhat_in_pf / np.linalg.norm(vhat_in_pf)

# Transformación a marco inercial heliocéntrico
rhat_in_ine  = M_euler @ rhat_in_pf
rhat_out_ine = M_euler @ rhat_out_pf
vhat_in_ine  = M_euler @ vhat_in_pf

# Dirección "desde dónde llegó": opuesta a la velocidad de entrada
u_from = -vhat_in_ine / np.linalg.norm(vhat_in_ine)

def ra_dec(u):
    ux, uy, uz = u / np.linalg.norm(u)
    ra = np.degrees(np.arctan2(uy, ux)) % 360.0
    dec = np.degrees(np.arcsin(uz))
    return ra, dec

ra_from, dec_from = ra_dec(u_from)

print(f"e = {e:.9f}")
print(f"psi (cos psi = 1/e) = {np.degrees(psi):.6f} deg")
print(f"f_inf = {np.degrees(f_inf):.6f} deg")
print("\nVector asintótico de entrada:", rhat_in_ine)
print("Vector asintótico de salida:", rhat_out_ine)
print("Dirección de llegada desde espacio profundo u_from:", u_from)
print(f"Radiant de llegada (RA, Dec) = ({ra_from:.6f} deg, {dec_from:.6f} deg)")

e = 1.205541073
psi (cos psi = 1/e) = 33.952277 deg
f_inf = 146.047723 deg

Vector asintótico de entrada: [ 0.13030606 -0.53808449  0.83275771]
Vector asintótico de salida: [0.90815065 0.13445231 0.39646561]
Dirección de llegada desde espacio profundo u_from: [ 0.13030606 -0.53808449  0.83275771]
Radiant de llegada (RA, Dec) = (283.613049 deg, 56.383073 deg)


## Visualización de la Trayectoria